In [1]:
import json

INPUT_PATH = "/mnt/data1/Ryunho/DenseRewardRLHF-PPO/news_pref_rm.jsonl"          # 원본 파일
OUTPUT_PATH = "/mnt/data1/Ryunho/DenseRewardRLHF-PPO/news_pref_rm_parsed.jsonl"  # 변환된 파일 이름


def main():
    num_lines = 0
    num_skipped = 0

    with open(INPUT_PATH, "r", encoding="utf-8") as fin, \
         open(OUTPUT_PATH, "w", encoding="utf-8") as fout:

        for line_no, line in enumerate(fin, start=1):
            line = line.strip()
            if not line:
                continue

            try:
                obj = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"[line {line_no}] JSON decode error: {e}")
                num_skipped += 1
                continue

            # 1) 기본 필드 체크
            if not all(k in obj for k in ["chosen", "rejected", "chosen_score", "rejected_score"]):
                print(f"[line {line_no}] Missing keys, got keys: {list(obj.keys())}")
                num_skipped += 1
                continue

            # 2) 이중 인코딩 풀기: 문자열 → 리스트[{"role", "content"}, ...]
            try:
                chosen_msgs = json.loads(obj["chosen"])
                rejected_msgs = json.loads(obj["rejected"])
            except Exception as e:
                print(f"[line {line_no}] Failed to parse chosen/rejected as JSON: {e}")
                num_skipped += 1
                continue

            # 3) 빈 대화는 스킵
            if len(chosen_msgs) == 0 or len(rejected_msgs) == 0:
                print(f"[line {line_no}] Empty chosen/rejected messages, skip.")
                num_skipped += 1
                continue

            # 4) margin 계산 (chosen_score - rejected_score)
            try:
                chosen_score = float(obj["chosen_score"])
                rejected_score = float(obj["rejected_score"])
            except Exception as e:
                print(f"[line {line_no}] Invalid score type: {e}")
                num_skipped += 1
                continue

            margin = chosen_score - rejected_score

            # 5) 최종 출력 포맷
            out_obj = {
                "chosen": chosen_msgs,            # 이제 리스트 형태
                "rejected": rejected_msgs,
                "chosen_score": chosen_score,
                "rejected_score": rejected_score,
                "margin": margin,
            }

            fout.write(json.dumps(out_obj, ensure_ascii=False) + "\n")
            num_lines += 1

    print(f"✅ Done! wrote {num_lines} lines to {OUTPUT_PATH}")
    if num_skipped > 0:
        print(f"⚠️ Skipped {num_skipped} problematic lines.")


if __name__ == "__main__":
    main()


✅ Done! wrote 10602 lines to /mnt/data1/Ryunho/DenseRewardRLHF-PPO/news_pref_rm_parsed.jsonl


In [1]:
import json

in_path = "news_pref_rm_parsed.jsonl"
out_path = "news_pref_rm_parsed_clean.jsonl"

def is_good_msg_list(msg_list):
    for m in msg_list:
        # content가 문자열이 아니면 (NaN, 숫자 등) 버린다
        if not isinstance(m.get("content", ""), str):
            return False
    return True

good = 0
bad = 0

with open(in_path, "r", encoding="utf-8") as fin, \
     open(out_path, "w", encoding="utf-8") as fout:
    for line in fin:
        if not line.strip():
            continue
        obj = json.loads(line)

        chosen = obj.get("chosen", [])
        rejected = obj.get("rejected", [])

        if is_good_msg_list(chosen) and is_good_msg_list(rejected):
            fout.write(json.dumps(obj, ensure_ascii=False) + "\n")
            good += 1
        else:
            bad += 1

print("good samples:", good)
print("dropped (bad) samples:", bad)


good samples: 10594
dropped (bad) samples: 8


In [2]:
import os
import pandas as pd

# CSV들이 들어있는 폴더 경로
folder_path = r"/mnt/data1/Ryunho/DenseRewardRLHF-PPO/gen_rl"   # ← 여기에 너의 CSV 폴더 경로 넣기

# 폴더 안에서 .csv 파일만 골라오기
csv_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

print("병합 대상 파일:")
for f in csv_files:
    print(" -", f)

df_list = []

# 모든 CSV 읽어서 리스트에 추가
for file in csv_files:
    file_path = os.path.join(folder_path, file)
    try:
        df = pd.read_csv(file_path)
        df_list.append(df)
    except Exception as e:
        print(f"⚠️ {file} 읽는 중 오류 발생:", e)

# 병합
merged_df = pd.concat(df_list, ignore_index=True)

# 저장
output_path = os.path.join(folder_path, "merged_all_news.csv")
merged_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("\n✅ 병합 완료!")
print("저장 경로:", output_path)
print("총 행 개수:", len(merged_df))

병합 대상 파일:
 - GPT_4.1-Gemma_3_27b-Eng_sc_final.csv
 - GPT_5-Gemma_3_27b-Eng_sc_final.csv
 - GPT_o1-Gemma_3_27b-Eng_sc_final.csv
 - GPT_5-Gemma_3_27b-Eng_we_final.csv
 - GPT_4o-Gemma_3_27b-Eng_ng_final.csv
 - GPT_o3-Gemma_3_27b-Eng_we_final.csv
 - GPT_4.1-Gemma_3_27b-Eng_ng_final.csv
 - GPT_o3-Gemma_3_27b-Eng_ng_final.csv
 - GPT_o3-Gemma_3_27b-Eng_sc_final.csv
 - GPT_o1-Gemma_3_27b-Eng_we_final.csv
 - GPT_4.1-Gemma_3_27b-Eng_we_final.csv
 - GPT_5-Gemma_3_27b-Eng_ng_final.csv
 - GPT_4o-Gemma_3_27b-Eng_we_final.csv
 - GPT_o1-Gemma_3_27b-Eng_ng_final.csv
 - GPT_4o-Gemma_3_27b-Eng_sc_final.csv

✅ 병합 완료!
저장 경로: /mnt/data1/Ryunho/DenseRewardRLHF-PPO/gen_rl/merged_all_news.csv
총 행 개수: 10602


In [4]:
import pandas as pd
import os

# 1. 병합된 CSV 불러오기
folder_path = r"/mnt/data1/Ryunho/DenseRewardRLHF-PPO/gen_rl"  # merged_all_news.csv가 있는 폴더
merged_path = os.path.join(folder_path, "merged_all_news.csv")

df = pd.read_csv(merged_path)

# 컬럼 이름 확인 (한 번만 찍어보고 주석 처리해도 됨)
print("컬럼들:", df.columns.tolist())

# 2. x 컬럼만 가져오기 (input prompt)
#   만약 컬럼 이름이 'x'가 아니라면, 실제 이름으로 바꿔줘야 함
prompts = df["x (Input prompt)"].dropna()

# 3. 공백/중복 제거
prompts = prompts.astype(str).str.strip()
prompts = prompts[prompts != ""]
prompts = prompts.drop_duplicates()

print("프롬프트 개수:", len(prompts))

# 4. PPO에서 쓰기 좋게 'prompt'라는 이름의 컬럼으로 만들기
prompt_df = pd.DataFrame({"prompt": prompts})

# JSONL로 저장 (DenseRewardRLHF에서 다루기 편하게)
output_jsonl = os.path.join(folder_path, "news_prompts.jsonl")
prompt_df.to_json(output_jsonl, orient="records", lines=True, force_ascii=False)

print("✅ 프롬프트 JSONL 저장 완료:", output_jsonl)


컬럼들: ['x (Input prompt)', 'y1 (GPT_4.1)', 'y2 (Gemma)', 'y1 (GPT_5)', 'y1 (GPT_o1)', 'y1 (GPT_4o)', 'y1 (GPT_o3)', 'y1 (GPT5)', 'pseudolabel']
프롬프트 개수: 8141
✅ 프롬프트 JSONL 저장 완료: /mnt/data1/Ryunho/DenseRewardRLHF-PPO/gen_rl/news_prompts.jsonl


In [ ]:
import json

in_path = "/mnt/data1/Ryunho/DenseRewardRLHF-PPO/gen_rl/news_prompts.jsonl"                # 기존 파일
out_path = "news_prompts_ppo.jsonl"           # 새로 만들 파일

with open(in_path, "r", encoding="utf-8") as fin, \
     open(out_path, "w", encoding="utf-8") as fout:
    for line in fin:
        if not line.strip():
            continue
        data = json.loads(line)
        
        # 기존 prompt는 그대로 두고
        prompt = data["prompt"]
        
        # chosen이 없으면 더미 추가
        if "chosen" not in data:
            data["chosen"] = [
                {
                    "role": "assistant",
                    "content": ""  # 빈 답변
                }
            ]
        
        fout.write(json.dumps(data, ensure_ascii=False) + "\n")

print("Done ->", out_path)


Done -> news_prompts_ppo.jsonl


: 